### Reading the file



In [1]:
import pandas as pd
df = pd.read_csv('../Data/SBAnational.csv',on_bad_lines='skip')

C:\Users\Akhil M\AppData\Local\Temp\ipykernel_23036\2538448538.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/SBAnational.csv',on_bad_lines='skip')


In [2]:
df.describe()

,LoanNr_ChkDgt,Zip,NAICS,Term,NoEmp,NewExist,CreateJob,RetainedJob,FranchiseCode,UrbanRural
count,8.991640e+05,899164.000000,899164.000000,899164.000000,899164.000000,899028.000000,899164.000000,899164.000000,899164.000000,899164.000000
mean,4.772612e+09,53804.391241,398660.950146,110.773078,11.411353,1.280404,8.430376,10.797257,2753.725933,0.757748
std,2.538175e+09,31184.159152,263318.312760,78.857305,74.108196,0.451750,236.688165,237.120600,12758.019136,0.646436
min,1.000014e+09,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.589758e+09,27587.000000,235210.000000,60.000000,2.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,4.361439e+09,55410.000000,445310.000000,84.000000,4.000000,1.000000,0.000000,1.000000,1.000000,1.000000
75%,6.904627e+09,83704.000000,561730.000000,120.000000,10.000000,2.000000,1.000000,4.000000,1.000000,1.000000
max,9.996003e+09,99999.000000,928120.000000,569.000000,9999.000000,2.000000,8800.000000,9500.000000,99999.000000,2.000000


In [3]:
df.info(verbose=True,show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899164 entries, 0 to 899163
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   LoanNr_ChkDgt      899164 non-null  int64  
 1   Name               899150 non-null  object 
 2   City               899134 non-null  object 
 3   State              899150 non-null  object 
 4   Zip                899164 non-null  int64  
 5   Bank               897605 non-null  object 
 6   BankState          897598 non-null  object 
 7   NAICS              899164 non-null  int64  
 8   ApprovalDate       899164 non-null  object 
 9   ApprovalFY         899164 non-null  object 
 10  Term               899164 non-null  int64  
 11  NoEmp              899164 non-null  int64  
 12  NewExist           899028 non-null  float64
 13  CreateJob          899164 non-null  int64  
 14  RetainedJob        899164 non-null  int64  
 15  FranchiseCode      899164 non-null  int64  
 16  Ur

### Split as Target and Input

In [4]:
Y=df['MIS_Status']
df.drop(columns=['MIS_Status'],inplace=True)   



In [5]:
df.isnull().mean().sort_values(ascending=False)

ChgOffDate           0.819055
RevLineCr            0.005036
LowDoc               0.002872
DisbursementDate     0.002634
BankState            0.001742
Bank                 0.001734
NewExist             0.000151
City                 0.000033
State                0.000016
Name                 0.000016
ApprovalFY           0.000000
ApprovalDate         0.000000
NAICS                0.000000
Zip                  0.000000
LoanNr_ChkDgt        0.000000
CreateJob            0.000000
NoEmp                0.000000
Term                 0.000000
UrbanRural           0.000000
FranchiseCode        0.000000
RetainedJob          0.000000
DisbursementGross    0.000000
BalanceGross         0.000000
ChgOffPrinGr         0.000000
GrAppv               0.000000
SBA_Appv             0.000000
dtype: float64

Check if final data is imbalanced. Apply SMOTE is ratio 20:1

In [6]:
Y.value_counts()

MIS_Status
P I F     739609
CHGOFF    157558
Name: count, dtype: int64

### Dropping more columns because they are causing data leakage

In [7]:
X=df.drop(columns=['ChgOffPrinGr','BalanceGross','ChgOffDate','Name'])


### Train Test splitting

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

### Split your Xtrain data to two different types of categories. Numerical vs Categorical. You further have to split Categorical as Nominal and Ordinal

In [9]:
numCols=list(X_train.select_dtypes(include=['int64','float64']).columns)
catCols=list(X_train.select_dtypes(include=['object']).columns)

In [10]:
numCols



['LoanNr_ChkDgt',
 'Zip',
 'NAICS',
 'Term',
 'NoEmp',
 'NewExist',
 'CreateJob',
 'RetainedJob',
 'FranchiseCode',
 'UrbanRural']

### Sending correct column to correct array

In [11]:
wrongCat=['DisbursementGross','GrAppv','SBA_Appv','DisbursementDate','ApprovalDate','ApprovalFY']
wrongNum=['NewExist','UrbanRural']

### Removing wrong data and appending it correctly to numCols

In [12]:
for col in wrongCat:
    catCols.remove(col)
numCols.extend(wrongCat)



### Removing wrong data from numCols and appending it correctly to catCols

In [13]:
for cols in wrongNum:
    numCols.remove(cols)
catCols.extend(wrongNum)

In [14]:
catCols

['City',
 'State',
 'Bank',
 'BankState',
 'RevLineCr',
 'LowDoc',
 'NewExist',
 'UrbanRural']

## Imputing missing values

### Numerical

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

In [16]:
df[numCols].isnull().mean().sort_values(ascending=False)

DisbursementDate     0.002634
LoanNr_ChkDgt        0.000000
NAICS                0.000000
Term                 0.000000
NoEmp                0.000000
Zip                  0.000000
CreateJob            0.000000
RetainedJob          0.000000
DisbursementGross    0.000000
FranchiseCode        0.000000
GrAppv               0.000000
SBA_Appv             0.000000
ApprovalDate         0.000000
ApprovalFY           0.000000
dtype: float64

Got concluded that there is no missing values in numerical columns

In [17]:
numImpute=SimpleImputer(strategy='mean')
colScale=StandardScaler()
numPipeline = Pipeline(steps=[
    ('numImp',numImpute),
    ('numScaling',colScale)
])

### Categorical

In [18]:
df[catCols].isnull().mean().sort_values(ascending=False)

RevLineCr     0.005036
LowDoc        0.002872
BankState     0.001742
Bank          0.001734
NewExist      0.000151
City          0.000033
State         0.000016
UrbanRural    0.000000
dtype: float64

### Split the Categorical into Nominal and Ordinal before encoding

In [19]:
X_train[numCols].dtypes


LoanNr_ChkDgt         int64
Zip                   int64
NAICS                 int64
Term                  int64
NoEmp                 int64
CreateJob             int64
RetainedJob           int64
FranchiseCode         int64
DisbursementGross    object
GrAppv               object
SBA_Appv             object
DisbursementDate     object
ApprovalDate         object
ApprovalFY           object
dtype: object

There are no ordinal columns so no need to apply ordinal encoding. Only apply one hot encoder

There are missing values in categorical hence impute using strategy=most-frequent

In [20]:
ohe=OneHotEncoder(handle_unknown='ignore')

In [21]:
catImpute=SimpleImputer(strategy='most_frequent')
catPipeline=Pipeline(steps=[
    ('catImp',catImpute),
    ('catEncoding',ohe),
    
])



Need to parse the data in numerical column

In [22]:
for df in [X_train, X_test]:
    df['ApprovalDate'] = pd.to_datetime(df['ApprovalDate'], errors='coerce')

    df['ApprovalMonth'] = df['ApprovalDate'].dt.month
    df['ApprovalYear'] = df['ApprovalDate'].dt.year
    df['ApprovalDay'] = df['ApprovalDate'].dt.day

C:\Users\Akhil M\AppData\Local\Temp\ipykernel_23036\1773141424.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['ApprovalDate'] = pd.to_datetime(df['ApprovalDate'], errors='coerce')
C:\Users\Akhil M\AppData\Local\Temp\ipykernel_23036\1773141424.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['ApprovalDate'] = pd.to_datetime(df['ApprovalDate'], errors='coerce')


In [23]:
for df in [X_train, X_test]:
    df['DisbursementDate'] = pd.to_datetime(df['DisbursementDate'], errors='coerce')

    df['DisbursementMonth'] = df['DisbursementDate'].dt.month
    df['DisbursementYear'] = df['DisbursementDate'].dt.year
    df['DisbursementDay'] = df['DisbursementDate'].dt.day

    df['ApprovalYYear'] = pd.to_numeric(df['ApprovalFY'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')


C:\Users\Akhil M\AppData\Local\Temp\ipykernel_23036\1258913485.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DisbursementDate'] = pd.to_datetime(df['DisbursementDate'], errors='coerce')
C:\Users\Akhil M\AppData\Local\Temp\ipykernel_23036\1258913485.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DisbursementDate'] = pd.to_datetime(df['DisbursementDate'], errors='coerce')


In [24]:
removeDate=['DisbursementDate','ApprovalDate','ApprovalFY']
for date in removeDate:
    numCols.remove(date)

dateCols=['DisbursementMonth','DisbursementYear','DisbursementDay','ApprovalYYear','ApprovalMonth','ApprovalDay','ApprovalYear']
numCols.extend(dateCols)


In [25]:
for df in [X_train, X_test]:
    df['GrAppv']=pd.to_numeric(df['GrAppv'].astype(str).str.replace(r'[\ $ ,]', '' ),errors='coerce')
    df['DisbursementGross']=pd.to_numeric(df['DisbursementGross'].astype(str).str.replace(r'[\ $ ,]', ''), errors='coerce')
    df['SBA_Appv']=pd.to_numeric(df['SBA_Appv'].astype(str).str.replace(r'[\ $ , ]', ''), errors='coerce')

In [26]:
df[catCols].dtypes

City           object
State          object
Bank           object
BankState      object
RevLineCr      object
LowDoc         object
NewExist      float64
UrbanRural      int64
dtype: object

### Transformer

In [27]:
from sklearn.compose import ColumnTransformer

In [28]:
processed=ColumnTransformer(transformers=[
    ('numProcessed',numPipeline,numCols),
    ('catProcessed',catPipeline,catCols)
])

In [29]:
X_train_processed=processed.fit_transform(X_train)
X_test_processed=processed.transform(X_test)

c:\Users\Akhil M\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['DisbursementGross' 'GrAppv' 'SBA_Appv']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
c:\Users\Akhil M\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: ['DisbursementGross' 'GrAppv' 'SBA_Appv']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


### Model Testing

In [30]:
from sklearn.linear_model import LogisticRegression

In [31]:
model=LogisticRegression()
model.fit(X_train_processed,Y_train)

ValueError: Input contains NaN

In [32]:
X_train[numCols].isna().sum().sort_values(ascending=False)

GrAppv               719331
DisbursementGross    719331
SBA_Appv             719331
DisbursementYear       1867
DisbursementDay        1867
DisbursementMonth      1867
Zip                       0
LoanNr_ChkDgt             0
NAICS                     0
Term                      0
RetainedJob               0
FranchiseCode             0
NoEmp                     0
CreateJob                 0
ApprovalYYear             0
ApprovalMonth             0
ApprovalDay               0
ApprovalYear              0
dtype: int64